# 01 Run Benchmark

Run a sweep and generate benchmark figures + summaries.

Default workflow is now the focused chain-bandit benchmark for the mean/variance question, with automatic all-core multiprocessing on this Mac.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

def _logical_cpus() -> int:
    try:
        return int(subprocess.check_output(["sysctl", "-n", "hw.logicalcpu"], text=True).strip())
    except Exception:
        return int(os.cpu_count() or 1)

LOGICAL_CPUS = max(1, _logical_cpus())
for key in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[key] = "1"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Logical CPUs:", LOGICAL_CPUS)
print("BLAS/OpenMP threads per worker: 1")


In [ ]:
import pandas as pd

from ess_ope.evaluation.sweep import SweepConfig, run_sweep
from ess_ope.evaluation.summary import (
    PaperClaimConfig,
    SummaryConfig,
    build_chain_bandit_sensitivity_summary,
    build_condition_summary,
    build_estimator_summary,
    build_paper_claim_summary,
    build_paper_claims_table,
)
from ess_ope.plotting.benchmark_figures import generate_benchmark_figures
from ess_ope.utils.config import load_yaml

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)


## Choose config
- `chain_bandit_focused.yaml`: recommended default for the PI mean/variance question
- `chain_bandit_baseline.yaml`: much wider chain-bandit sweep
- `random_mdp_robust.yaml` / `random_mdp_ultra.yaml`: original sequential benchmark


In [ ]:
cfg_path = PROJECT_ROOT / "configs/sweeps/chain_bandit_focused.yaml"
cfg = SweepConfig.from_dict(load_yaml(cfg_path))

# Use all reported logical CPUs on this M3 Pro, with single-threaded workers.
cfg.num_workers = LOGICAL_CPUS
cfg.mp_chunksize = 4

print("Config:", cfg_path.name)
print("env_name:", cfg.env_name)
print("num_workers:", cfg.num_workers)
print("mp_chunksize:", cfg.mp_chunksize)
cfg


In [ ]:
# Optional smoke override for fast notebook validation.
USE_SMOKE_OVERRIDE = False
if USE_SMOKE_OVERRIDE:
    cfg.name = f"{cfg.name}_smoke"
    cfg.seeds = [0, 1]
    cfg.alphas = [0.1, 0.6]
    cfg.betas = [0.0, 0.6]
    cfg.dataset_sizes = [100]
    cfg.env_repeats = 1
    cfg.policy_repeats = 1
    cfg.dataset_repeats = 1
    if cfg.env_name == "chain_bandit":
        cfg.transition_strengths = [0.0, 1.0]
        cfg.reward_mean_scales = [0.5, 2.0]
        cfg.reward_gaps = [0.5]
        cfg.reward_stds = [0.1, 1.0]
        cfg.chain_variants = ["reward_only", "transitional"]

cfg


In [ ]:
df, run_dir, result_path = run_sweep(cfg)
fig_dir = run_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

benchmark_report = generate_benchmark_figures(df, output_dir=fig_dir, fixed_alpha=None, fixed_beta=0.0)
summary_cfg = SummaryConfig(bootstrap_samples=16000, bootstrap_max_points=200000, show_progress=True)
estimator_summary = build_estimator_summary(df, config=summary_cfg)
condition_summary = build_condition_summary(df, show_progress=True)
chain_bandit_sensitivity = build_chain_bandit_sensitivity_summary(df)
paper_claims = build_paper_claims_table(df, estimator_summary, benchmark_report, config=PaperClaimConfig())
paper_claim_summary = build_paper_claim_summary(paper_claims)

benchmark_report.to_csv(fig_dir / "benchmark_report.csv", index=False)
estimator_summary.to_csv(fig_dir / "estimator_summary.csv", index=False)
condition_summary.to_csv(fig_dir / "condition_summary.csv", index=False)
if not chain_bandit_sensitivity.empty:
    chain_bandit_sensitivity.to_csv(fig_dir / "chain_bandit_sensitivity_summary.csv", index=False)
paper_claims.to_csv(fig_dir / "paper_claims.csv", index=False)
paper_claim_summary.to_csv(fig_dir / "paper_claim_summary.csv", index=False)

print("Run dir:", run_dir)
print("Results:", result_path)
print("Figure dir:", fig_dir)
print("Rows:", len(df))
chain_bandit_sensitivity if not chain_bandit_sensitivity.empty else paper_claims
